# Missing Values

### Learning Objectives

We learn to:

1. Identify causes of missing data
2. Apply `KNNImputer` to fill missing values; both numerical and categorical
3. Detect errors where `fit` and `transform` are misused on the wrong set of data

## The Problem

Most library algorithms require that the data contain no gaps (**missing values**).

There are many reasons why data is often missing:

- **Human error:** Accidental omission of input, or the user deliberately not answering to protect their privacy.
- **Technical failure:** Malfunctioning sensors or connection drops during data transfer.
- **Data merging:** Gaps from join operations (e.g. SQL joins) or changes in table structure over time.
- **Attrition:** Data loss when participants drop out of longitudinal studies or delete their accounts.
    - Example: In clinical trials, a patient may drop out due to migration, death, loss of interest in follow-up, worsening condition, or severe side effects.
    - Example: In subscription-based systems, these are customers who stopped using the service; their data appears empty in the months after they churned.

## Sample Data

- We use a small set of health measurements (age, systolic blood pressure, cholesterol, body mass index) with missing values.
- We first see what the data looks like, then what happens if we drop rows that contain missing values.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer

In [2]:
# Realistic data: patient measurements (age, systolic BP mmHg, cholesterol mg/dL, BMI)
np.random.seed(42)
n = 12
age = np.random.randint(35, 70, size=n)
systolic_bp = 100 + 0.5 * age + np.random.randn(n) * 8  # rough relationship with age
cholesterol = 150 + 0.8 * age + np.random.randn(n) * 15
bmi = 22 + 0.1 * age + np.random.randn(n) * 2

df = pd.DataFrame({
    "age": age.astype(float),
    "blood_pressure": systolic_bp,
    "cholesterol": cholesterol,
    "bmi": bmi,
})

# Introduce missing values realistically (e.g. one patient missing BP, another missing cholesterol)
df.loc[1, "blood_pressure"] = np.nan
df.loc[3, "cholesterol"] = np.nan
df.loc[5, "age"] = np.nan
df.loc[7, "bmi"] = np.nan
df.loc[9, "blood_pressure"] = np.nan
df.loc[9, "cholesterol"] = np.nan

print("Data with missing values (NaN):")
display(df)

Data with missing values (NaN):


,age,blood_pressure,cholesterol,bmi
0,63.0,132.138654,192.523157,25.544121
1,49.0,NaN,217.891569,26.193767
2,42.0,121.177775,153.199206,25.277069
3,55.0,124.077657,NaN,27.633315
4,53.0,122.245461,204.087890,26.947429
5,NaN,127.560196,179.083534,30.101786
6,45.0,124.276631,202.953423,27.896798
7,45.0,116.356188,191.596784,NaN
8,58.0,130.139717,190.602906,25.985627
9,58.0,NaN,NaN,30.177251


### Naive Solution

The naive approach is to remove rows that have missing values.

This leads to loss of the other information in those rows. For example: missing the `age` in a row while have all three others features: `blood_pressure`, `cholesterol`, `bmi` would simply remove them from our dataset.

This can be done in the DataFrame using the `dropna()` method as follows:

In [3]:
df_dropped = df.dropna()
print(f"Original number of rows: {len(df)}")
print(f"After dropping rows with missing values: {len(df_dropped)} rows")
print("→ We lost information from entire rows because of a single missing value per row.")
display(df_dropped)

Original number of rows: 12
After dropping rows with missing values: 7 rows
→ We lost information from entire rows because of a single missing value per row.


,age,blood_pressure,cholesterol,bmi
0,63.0,132.138654,192.523157,25.544121
2,42.0,121.177775,153.199206,25.277069
4,53.0,122.245461,204.087890,26.947429
6,45.0,124.276631,202.953423,27.896798
8,58.0,130.139717,190.602906,25.985627
10,37.0,127.574714,188.091692,27.271064
11,56.0,127.162036,184.233198,32.912020


## Imputation Methods

A better approach is to **impute** (fill-in) the missing values—i.e., **infer them from the known part of the data**.

We can infer them in two main ways:

- **Univariate:** From other values of the same variable ([`SimpleImputer`](https://scikit-learn.org/stable/modules/impute.html#univariate-feature-imputation))—e.g. mean, median, most frequent, or a constant.
- **Multivariate:** Using other variables (e.g. [`IterativeImputer`](https://scikit-learn.org/stable/modules/impute.html#multivariate-feature-imputation)), treating it as a prediction task: regression or classification depending on the type of the variable with the missing value.

## First: Simple Imputation (SimpleImputer)

Impute missing values using only the same variable: mean, median, most frequent, or a constant.

In [4]:
X = df.values

mean_imputer = SimpleImputer(strategy="mean")
X_mean = mean_imputer.fit_transform(X)
df_mean = pd.DataFrame(X_mean, columns=df.columns, index=df.index)
print("After imputation with mean:")
display(df_mean.round(2))

After imputation with mean:


,age,blood_pressure,cholesterol,bmi
0,63.0,132.14,192.52,25.54
1,49.0,125.27,217.89,26.19
2,42.0,121.18,153.20,25.28
3,55.0,124.08,190.43,27.63
4,53.0,122.25,204.09,26.95
5,51.0,127.56,179.08,30.10
6,45.0,124.28,202.95,27.90
7,45.0,116.36,191.60,27.81
8,58.0,130.14,190.60,25.99
9,58.0,125.27,190.43,30.18


In [5]:
median_imputer = SimpleImputer(strategy="median")
X_median = median_imputer.fit_transform(X)
df_median = pd.DataFrame(X_median, columns=df.columns, index=df.index)
print("After imputation with median:")
display(df_median.round(2))

After imputation with median:


,age,blood_pressure,cholesterol,bmi
0,63.0,132.14,192.52,25.54
1,49.0,125.72,217.89,26.19
2,42.0,121.18,153.20,25.28
3,55.0,124.08,191.10,27.63
4,53.0,122.25,204.09,26.95
5,53.0,127.56,179.08,30.10
6,45.0,124.28,202.95,27.90
7,45.0,116.36,191.60,27.27
8,58.0,130.14,190.60,25.99
9,58.0,125.72,191.10,30.18


## Second: K-Nearest Neighbors Imputation (KNNImputer)

Impute each missing value using the nearest samples in terms of distance (missing values are ignored when computing distances).

### K-NN Classifier

A **k-NN classifier**, the output of which is a class membership. An object is classified by a plurality vote of its K-neighbors:

![Figure: Application of a k-NN classifier considering k = 3 neighbors.](https://upload.wikimedia.org/wikipedia/commons/7/78/KNN_decision_surface_animation.gif)

The figure above shows application of a k-NN classifier considering k = 3 neighbors.

- **Left** - Given the test point "?", the algorithm seeks the 3 closest points in the training set, and adopts the majority vote to classify it as "class red".
- **Right** - By iteratively repeating the prediction over the whole feature space (X1, X2), one can depict the "decision surface"

**Usage**: K-NN classifier can be used to fill-in **categorical variables**.

### K-NN Regression

In **k-NN regression**, also known as **k-NN smoothing**; used for **continuous variables**, it uses a weighted **average of the k nearest neighbors**, weighted by the inverse of their distance.

![Figure: imputation of continuous values](../assets/impute_continuous_value.png)

This can be used to fill any missing point in the data:

::: {.callout-warning}

Distanced-based algorithms like KNN assume scaled numerical features. Otherwise, you will get erroneous results!

:::

We will first scaled our features using `StandardScaler`; which makes the values have a mean of zero and standard deviation of 1 (more on this later):

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X)  # Fit on mean-imputed data to get scaling parameters
X_scaled = scaler.transform(X)  # Scale the mean-imputed data for KNN

Now we impute the scaled data:

In [ ]:
knn_imputer = KNNImputer(n_neighbors=3)
X_knn = knn_imputer.fit_transform(X_scaled)

And show the resulting DataFrame with filled values; but we need to **inverse the scaling** to get the original unscaled numbers:

In [ ]:

print("After imputation with KNN (n_neighbors=3):")
df_knn = pd.DataFrame(
    scaler.inverse_transform(X_knn),
    columns=df.columns,
    index=df.index
)
display(df_knn.round(2))


After imputation with KNN (n_neighbors=3):


,age,blood_pressure,cholesterol,bmi
0,63.00,132.14,192.52,25.54
1,49.00,123.53,217.89,26.19
2,42.00,121.18,153.20,25.28
3,55.00,124.08,208.31,27.63
4,53.00,122.25,204.09,26.95
5,50.33,127.56,179.08,30.10
6,45.00,124.28,202.95,27.90
7,45.00,116.36,191.60,27.01
8,58.00,130.14,190.60,25.99
9,58.00,126.27,189.13,30.18




We talk about **feature scaling** next.

---

- For more, see [7.4. Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html).